# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Print metadata summary
meta = dataset.metadata
print(meta)

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's list all record sets, then the fields available in each set, referencing all entities by their `@id`.

In [ ]:
# List all available record sets and fields by @id

record_sets = dataset.metadata.record_sets

print(f"Found {len(record_sets)} Record Sets:")
for rs in record_sets:
    print(f"- Record Set @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")
    print("  Fields:")
    for field in rs.get('field', []):
        # Field might be a @id or a dict
        if isinstance(field, dict):
            print(f"    - Field @id: {field.get('@id')}, name: {field.get('name', field.get('@id'))}")
        else:
            print(f"    - Field @id: {field}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We'll use the record set and field `@id`s identified above.

We'll load **all** available record sets and display their field IDs and the first few rows.

In [ ]:
# Load all data into DataFrames by Record Set @id
dataframes = {}
for rs in record_sets:
    rs_id = rs['@id']  # Always use @id for reference
    print(f'Loading records for Record Set: {rs_id}')
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f'Columns for {rs_id}:', df.columns.tolist())
        display(df.head())
    else:
        print(f'No records found for {rs_id}.')
    print('-' * 40)
# Store one non-empty record set for EDA/visualization
main_rs_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        main_rs_id = rs_id
        break

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping.

We'll select a numeric field for EDA. You may want to change the `numeric_field_id` and `group_field_id` based on the column list printed above.

In [ ]:
# ---
# Example: Replace these with actual @id of numeric field and group field found above
# You can inspect the columns in dataframes[main_rs_id].columns
df = dataframes[main_rs_id]
print(f'Record Set Used for EDA: {main_rs_id}')

columns = df.columns.tolist()
print('Columns available:', columns)

# Pick a numeric field (e.g., 'cr:field:Abundance') and a group field (e.g., 'cr:field:SampleType')
# Replace with actual IDs as needed
numeric_field = None
for c in columns:
    # Try to pick a likely numeric field
    if 'abundance' in c.lower() or 'amount' in c.lower() or 'coverage' in c.lower() or 'mw' in c.lower():
        numeric_field = c
        break

# Fallback if not found
if numeric_field is None:
    for c in columns:
        if df[c].dtype.kind in 'if' and not df[c].isnull().all():
            numeric_field = c
            break

if numeric_field is not None:
    print(f'Using numeric field: {numeric_field}')
    # Try a threshold based on field statistics
    if df[numeric_field].dtype.kind in 'i':
        threshold = df[numeric_field].quantile(0.9)  # Top 10%
    else:
        threshold = df[numeric_field].mean()

    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a string/categorical field
    group_field = None
    for c in columns:
        if c != numeric_field and df[c].dtype == object and df[c].nunique() < len(df) // 2:
            group_field = c
            break
    if group_field:
        print(f'Grouping by: {group_field}')
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        display(grouped_df.head())
else:
    print('No suitable numeric field found for EDA.')

## 5. Visualization
Visualize numeric distributions or grouped statistics using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'{numeric_field} Distribution')
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group if available
    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=30)
        plt.show()
else:
    print('No numeric field available for visualization.')

## 6. Conclusion
In this notebook, we:
- Loaded and explored the FAIR^2 mass spectrometry dataset using a Croissant schema
- Listed all record sets and field IDs for reference
- Extracted data into DataFrames for programmatic analysis
- Performed common EDA: filtering, normalization, and grouping on selected numeric fields
- Visualized distributions to aid further downstream analysis

You can further customize EDA or feature engineering steps by referencing the specific `@id`s of fields and record sets from the Croissant metadata above.